# 🧙‍♂️ Решение Задания 5: Детекция препятствий в подземных выработках на данных 3D-LiDAR
## Научный двухуровневый геометрический пайплайн (Two-Tier Frenet-Invariance Pipeline)

### 📌 Постановка задачи и соревновательная специфика
В горнодобывающей шахте беспилотная вагонетка перемещается по рельсовому пути, оборудованная 3D-LiDAR сенсором. Задача — обнаружить препятствия (валуны, обвалы породы, забытое оборудование, персонал) в габарите пути по 10-кадровым клипам лидарных облаков точек.

Целевая соревновательная метрика:
$$S = R \cdot P \cdot (1 - F)^2$$
где:
* $R = \frac{1}{3} (R_{0-10m} + R_{10-25m} + R_{25-60m})$ — макро-усредненный Recall по трем дистанционным диапазонам;
* $P$ — макро-усредненный Precision по кадрам с препятствиями (попадание хотя бы одной предсказанной точки в объект дает $100\%$ Recall для этого кадра!);
* $F$ — доля ложных тревог (False Alarm Rate) от всех возможных окон наблюдения (штрафуется квадратично!).

---

### 🔬 Ключевые научные открытия и архитектура двухуровневого пайплайна (Two-Tier Pipeline)
1. **Геометрическая неинвариантность статического габарита (The Static Box Fallacy):**
   * В подземных выработках рельсовый путь не является прямой линией: туннели изгибаются в горизонтальной плоскости ($y$ достигает $+12$ м) и имеют крутые уклоны вверх/вниз ($z$ изменяется на метры).
   * Статический габарит ($|y| \le 0.55$ м, $z \in [-0.69, 0.30]$ м) полностью теряет препятствия на поворотах и подъемах, ограничивая Recall потолком $\approx 0.50$.
   * **Решение:** Динамическое оценивание профиля туннеля и уровня полотна (`track_tunnel`) с переводом точек в **путевые координаты Френе** $(s, n, h)$:
     $$n = y - y_{\text{track}}(x) \quad (\text{боковое смещение относительно оси пути})$$
     $$h = z - z_{\text{floor}}(x) \quad (\text{высота над основанием рельсового полотна})$$
     Канонический габарит пути в координатах Френе: $|n| \le 0.48$ м, $h \in [0.20, 1.15]$ м.

2. **Защита от дренажных канав и потолочных отражений:**
   * Шахтные выработки имеют водоотводные канавы по бокам (где $z < -1.2$ м) и арочный свод потолка.
   * Алгоритм `track_tunnel` использует *slope-consistent tracking* с ограничением градиента уклона $|\Delta z / \Delta x| \le 0.15$ и оценкой пола строго в центральном створе между рельсами.

3. **Селектор ядерных точек (Top-$K$ Core Points Selector):**
   * По правилам соревнования предсказание даже **одной** точки на объекте дает кадру $100\%$ Recall.
   * Предсказание полного облака точек приводит к тому, что граничные точки объекта соприкасаются с рельсами, гравием или шпалами, роняя Precision кадра $P$.
   * Пайплайн выбирает ровно $K=2$ наиболее уверенные центральные точки ядра объекта для каждого кадра, что максимизирует Precision $P \to 0.90$ без потери Recall!

4. **Двухсетевой ансамбль PointNet (Cartesian + Frenet):**
   * Объединение базовой декартовой сети `model_3d_pointnet.pt` (высокая чувствительность на прямых участках) и специализированной геометрической сети `model_frenet_pointnet.pt` (инвариантность на поворотах и уклонах).


In [2]:
# 1. Импорт библиотек и проверка аппаратного ускорения
import os
import sys
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

# Добавляем пути к модулям
sys.path.append(str(Path('.').resolve()))
from evaluate import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


PyTorch version: 2.1.2+cu121
CUDA available: True
Active GPU: NVIDIA GeForce RTX 4090
VRAM: 24.00 GB


### 📐 Уровень 1: Динамическое оценивание оси пути и уровня полотна
Функция `track_tunnel` разбивает облако точек 10-кадрового клипа на продольные секторы по дистанции $x$ (от $0.5$ до $55$ м).
В каждом секторе:
1. Вычисляется центральная линия туннеля $y_{\text{track}}(x)$ как медиана между левой и правой стенами выработки (6-й и 94-й перцентили по $y$).
2. Оценивается уровень пола полотна $z_{\text{floor}}(x)$ среди точек в створе рельсов ($|y - y_{\text{track}}| \le 0.85$ м) с робастным сглаживанием уклона (slope tracking) для исключения дренажных канав и выбросов.


In [4]:
def track_tunnel(c_pts):
    """
    Робастная оценка центральной оси пути y_track(x) и уровня пола z_floor(x).
    Использует адаптивное секционирование и slope-consistent сглаживание уклона.
    """
    x, y, z = c_pts[:, 0], c_pts[:, 1], c_pts[:, 2]
    x_bins = [(0.5, 3), (3, 6), (6, 10), (10, 15), (15, 21), (21, 28), (28, 36), (36, 45), (45, 55)]
    xs, ys, zs, slopes_z = [0.0], [0.0], [-0.85], [0.0]
    
    for xl, xh in x_bins:
        xm = (xl + xh) / 2.0
        m = (x >= xl) & (x < xh)
        if np.sum(m) < 12: 
            continue
        p = c_pts[m]
        yl = np.percentile(p[:, 1], 6)
        yr = np.percentile(p[:, 1], 94)
        ymid = (yl + yr) / 2.0
        
        dx = xm - xs[-1]
        z_expected = zs[-1] + slopes_z[-1] * dx
        near_center = p[np.abs(p[:, 1] - ymid) <= 0.85]
        
        if len(near_center) >= 6:
            floor_cands = near_center[np.abs(near_center[:, 2] - z_expected) <= 0.6]
        else:
            floor_cands = p[np.abs(p[:, 2] - z_expected) <= 0.6]
            
        if len(floor_cands) >= 5:
            zf = np.percentile(floor_cands[:, 2], 12)
            curr_slope = np.clip((zf - zs[-1]) / dx, -0.15, 0.15)
            new_slope = 0.5 * slopes_z[-1] + 0.5 * curr_slope
        else:
            zf = z_expected
            new_slope = slopes_z[-1]
            
        xs.append(xm)
        ys.append(ymid)
        zs.append(zf)
        slopes_z.append(new_slope)
        
    return np.array(xs), np.array(ys), np.array(zs)

print("✅ Функция track_tunnel успешно скомпилирована.")


✅ Функция track_tunnel успешно скомпилирована.


### 🔄 Преобразование в путевые координаты Френе и экстракция признаков
Для каждой точки вычисляются инвариантные признаки относительно локального положения пути:
* $n = y - y_{\text{track}}(x)$ — боковое отклонение от центра колеи;
* $h = z - z_{\text{floor}}(x)$ — высота точки над уровнем рельсов;
* Геометрические зазоры до канонического габарита пути: $|n| \le 0.48$ м, $h \in [0.20, 1.15]$ м.


In [6]:
def extract_orig_features(pts):
    """Извлечение 13-мерного признакового вектора для декартовой модели."""
    x = pts[:, 0]
    y = pts[:, 1]
    z = pts[:, 2]
    r_xy = np.sqrt(x**2 + y**2)
    r_3d = np.sqrt(x**2 + y**2 + z**2)
    pitch = np.arctan2(z, r_xy)
    yaw = np.arctan2(y, x)
    abs_y = np.abs(y)
    h_est = z + 0.85
    in_box = ((abs_y <= 0.55) & (z >= -0.69) & (z <= 0.30)).astype(np.float32)
    feats = np.column_stack([
        x, y, z, r_xy, r_3d, pitch, yaw, abs_y, h_est, in_box,
        0.55 - abs_y, z - (-0.69), 0.30 - z
    ]).astype(np.float32)
    return feats

def extract_frenet_features(cand_pts, xs, ys, zs):
    """Извлечение 13-мерного признакового вектора в инвариантных координатах Френе."""
    x = cand_pts[:, 0]
    y = cand_pts[:, 1]
    z = cand_pts[:, 2]
    y_tr = np.interp(x, xs, ys)
    z_fl = np.interp(x, xs, zs)
    n = y - y_tr
    h = z - z_fl
    r_nh = np.sqrt(n**2 + h**2)
    r_3d = np.sqrt(x**2 + n**2 + h**2)
    pitch_fr = np.arctan2(h, np.sqrt(x**2 + n**2) + 1e-6)
    yaw_fr = np.arctan2(n, x + 1e-6)
    abs_n = np.abs(n)
    in_gauge = ((abs_n <= 0.50) & (h >= 0.18) & (h <= 1.15)).astype(np.float32)
    margin_n = 0.50 - abs_n
    margin_h_bot = h - 0.18
    margin_h_top = 1.15 - h
    feats = np.column_stack([
        x, n, h, r_nh, r_3d, pitch_fr, yaw_fr,
        abs_n, in_gauge, margin_n, margin_h_bot, margin_h_top,
        np.abs(h - 0.65)
    ]).astype(np.float32)
    return feats

print("✅ Функции извлечения признаков (декартовых и Френе) готовы.")


✅ Функции извлечения признаков (декартовых и Френе) готовы.


### 🧠 Уровень 2: Архитектура 3D-нейросети PointNet и загрузка весов
Используется архитектура `PointNetSegmenter` с многослойными перцептронами, LayerNorm, GELU-активациями и модулем глобального объединения признаков (Global Max + Global Average Pooling).
В ансамбле работают две модели:
1. `model_3d_pointnet.pt` — обученная на декартовых признаках.
2. `model_frenet_pointnet.pt` — обученная в канонических координатах Френе.


In [8]:
class PointNetSegmenter(nn.Module):
    def __init__(self, in_channels=13):
        super().__init__()
        self.mlp1 = nn.Sequential(
            nn.Linear(in_channels, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )
        self.global_mlp = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )
        self.seg_head = nn.Sequential(
            nn.Linear(256 + 256, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, N, C = x.shape
        point_feat = self.mlp1(x)
        max_feat = torch.max(point_feat, dim=1)[0]
        avg_feat = torch.mean(point_feat, dim=1)
        glob_feat = self.global_mlp(torch.cat([max_feat, avg_feat], dim=-1))
        glob_expand = glob_feat.unsqueeze(1).expand(-1, N, -1)
        fusion = torch.cat([point_feat, glob_expand], dim=-1)
        logits = self.seg_head(fusion)
        return logits.squeeze(-1)

# Загрузка моделей на GPU
orig_net = PointNetSegmenter(13).to(device)
orig_net.load_state_dict(torch.load('model_3d_pointnet.pt', map_location=device))
orig_net.eval()

frenet_net = PointNetSegmenter(13).to(device)
frenet_net.load_state_dict(torch.load('model_frenet_pointnet.pt', map_location=device))
frenet_net.eval()

print("🚀 Модели PointNet (Cartesian + Frenet) успешно загружены и переведены в eval-режим!")


🚀 Модели PointNet (Cartesian + Frenet) успешно загружены и переведены в eval-режим!


### 🎯 Двухуровневый алгоритм инференса и селектор ядерных точек (Top-$K$ Core Selector)
Для каждого клипа:
1. Выполняется отслеживание туннеля `track_tunnel`.
2. В каждом кадре вычисляются вероятности принадлежности точек препятствию от обеих сетей.
3. Точки фильтруются по каноническому габариту пути.
4. **Top-$K$ Core Selector:** Для кадра с подтвержденным препятствием выбираются ровно $K=2$ точки с максимальной уверенностью. Это гарантирует Precision кадра $P \approx 0.90$, так как исключаются размытые краевые точки на границах объекта.


In [10]:
# Оптимальные гиперпараметры, подобранные на валидационной выборке
THRESH_ORIG = 0.72       # Порог декартовой модели
THRESH_FRENET = 0.85     # Порог инвариантной Френе-модели
CORE_K = 2               # Количество центральных ядерных точек на кадр
MIN_ACTIVE_FRAMES = 1    # Минимальное количество подтвержденных кадров в клипе

print(f"Конфигурация гиперпараметров:")
print(f"  * THRESH_ORIG: {THRESH_ORIG}")
print(f"  * THRESH_FRENET: {THRESH_FRENET}")
print(f"  * CORE_K: {CORE_K} (Top-K Core Selector)")
print(f"  * MIN_ACTIVE_FRAMES: {MIN_ACTIVE_FRAMES}")


Конфигурация гиперпараметров:
  * THRESH_ORIG: 0.72
  * THRESH_FRENET: 0.85
  * CORE_K: 2 (Top-K Core Selector)
  * MIN_ACTIVE_FRAMES: 1


### 📊 Валидация двухуровневого пайплайна на выборке `val/`
Проводим расчет официальной соревновательной метрики $S = R \cdot P \cdot (1 - F)^2$ на всех 24 файлах валидации с помощью официального скрипта `evaluate.py`.


In [12]:
# Выполняем полный цикл инференса и валидации на val/
VAL_DIR = Path('val')
TEMP_PRED_VAL = Path('temp_hybrid.bin')

val_files = sorted(VAL_DIR.glob("*.npz"))
print(f"Запуск валидации на {len(val_files)} файлах выборки val/...")

val_preds = []
total_val_points = 0
t_start = time.time()

with torch.no_grad():
    for vf in val_files:
        with np.load(vf) as d:
            pts = d['points']
            offsets = d['frame_offsets']
            total_val_points += len(pts)
            
            p_orig = np.zeros(len(pts), dtype=np.float32)
            p_frenet = np.zeros(len(pts), dtype=np.float32)
            pred = np.zeros(len(pts), dtype=np.uint8)
            
            n_clips = (len(offsets) - 1) // 10
            for c in range(n_clips):
                lo, hi = offsets[c * 10], offsets[(c + 1) * 10]
                cp = pts[lo:hi]
                xs, ys, zs = track_tunnel(cp)
                
                clip_cands = []
                for f in range(10):
                    flo, fhi = offsets[c * 10 + f], offsets[c * 10 + f + 1]
                    fp = pts[flo:fhi]
                    fx, fy, fz = fp[:, 0], fp[:, 1], fp[:, 2]
                    
                    # Декартовая инференс-ветка
                    c_mask_orig = (np.abs(fy) <= 0.80) & (fz >= -0.80) & (fz <= 0.45) & (fx >= 0.2) & (fx <= 55.0)
                    if np.any(c_mask_orig):
                        feat_o = extract_orig_features(fp[c_mask_orig])
                        tx_o = torch.from_numpy(feat_o).unsqueeze(0).to(device)
                        with torch.amp.autocast('cuda'):
                            log_o = orig_net(tx_o).squeeze(0)
                            p_orig[flo:fhi][c_mask_orig] = torch.sigmoid(log_o).cpu().numpy()
                            
                    # Френе инференс-ветка
                    y_tr = np.interp(fx, xs, ys)
                    z_fl = np.interp(fx, xs, zs)
                    dy = np.abs(fy - y_tr)
                    dh = fz - z_fl
                    c_mask_fr = (fx >= 0.2) & (fx <= 58.0) & (dy <= 0.65) & (dh >= 0.08) & (dh <= 1.25)
                    if np.any(c_mask_fr):
                        feat_fr = extract_frenet_features(fp[c_mask_fr], xs, ys, zs)
                        tx_fr = torch.from_numpy(feat_fr).unsqueeze(0).to(device)
                        with torch.amp.autocast('cuda'):
                            log_fr = frenet_net(tx_fr).squeeze(0)
                            p_frenet[flo:fhi][c_mask_fr] = torch.sigmoid(log_fr).cpu().numpy()
                            
                    # Канонические границы
                    in_orig_box = (np.abs(fy) <= 0.55) & (fz >= -0.69) & (fz <= 0.30)
                    m_orig = (p_orig[flo:fhi] >= THRESH_ORIG) & in_orig_box
                    
                    in_fr_box = (dy <= 0.48) & (dh >= 0.20) & (dh <= 1.15)
                    m_fr = (p_frenet[flo:fhi] >= THRESH_FRENET) & in_fr_box
                    
                    m_union = m_orig | m_fr
                    clip_cands.append(np.where(m_union)[0])
                    
                active_f = [f for f in range(10) if len(clip_cands[f]) > 0]
                if len(active_f) >= MIN_ACTIVE_FRAMES:
                    for f in active_f:
                        flo = offsets[c * 10 + f]
                        cands = clip_cands[f]
                        score = np.maximum(p_orig[flo:offsets[c * 10 + f + 1]][cands], 
                                           p_frenet[flo:offsets[c * 10 + f + 1]][cands])
                        if len(cands) > CORE_K:
                            topk = np.argsort(score)[-CORE_K:]
                            pred[flo + cands[topk]] = 1
                        else:
                            pred[flo + cands] = 1
                            
            val_preds.append(pred)

# Упаковка и официальная оценка
packed_val = np.packbits(np.concatenate(val_preds), bitorder='little').tobytes()
with open(TEMP_PRED_VAL, 'wb') as f:
    f.write(packed_val)

val_time = time.time() - t_start
print(f"Валидационный инференс завершен за {val_time:.2f} с ({len(val_files)} файлов, {total_val_points:,} точек).")

# Расчет официальной метрики
res = evaluate(VAL_DIR, TEMP_PRED_VAL)
print("\n" + "="*60)
print("  РЕЗУЛЬТАТЫ ОЦЕНКИ ДВУХУРОВНЕВОГО НАУЧНОГО ПАЙПЛАЙНА")
print("="*60)
print(f"  🏆 Итоговый Score S : {res['S']:.6f}  (Baseline S = 0.429712, Прирост: +{(res['S']/0.429712 - 1)*100:.2f}%)")
print(f"  📈 Macro Recall   R : {res['R']:.6f}")
r_ranges = res.get('recall_by_range', {})
r_0_10 = r_ranges.get('0-10m') or 0.0
r_10_25 = r_ranges.get('10-25m') or 0.0
r_25_60 = r_ranges.get('25-60m') or 0.0
print(f"     - R [0-10m]      : {r_0_10:.6f}")
print(f"     - R [10-25m]     : {r_10_25:.6f}")
print(f"     - R [25-60m]     : {r_25_60:.6f}")
print(f"  🎯 Frame Precision P: {res['P']:.6f}")
print(f"  🛡️ False Alarm Rate: {res['F']:.6f}")
print("="*60)


Запуск валидации на 24 файлах выборки val/...
Валидационный инференс завершен за 28.45 с (24 файлов, 18,720,000 точек).

  РЕЗУЛЬТАТЫ ОЦЕНКИ ДВУХУРОВНЕВОГО НАУЧНОГО ПАЙПЛАЙНА
  🏆 Итоговый Score S : 0.474744  (Baseline S = 0.429712, Прирост: +10.48%)
  📈 Macro Recall   R : 0.552144
     - R [0-10m]      : 0.767015
     - R [10-25m]     : 0.742813
     - R [25-60m]     : 0.146604
  🎯 Frame Precision P: 0.896645
  🛡️ False Alarm Rate: 0.020732


### 🚀 Финальный инференс на тестовой выборке `test/` (1000 клипов)
Выполняем инференс научного пайплайна на всех 1000 тестовых файлах, упаковываем битовые маски и верифицируем результирующий бинарный файл `predictions.bin`.


In [14]:
TEST_DIR = Path('test')
OUT_PRED = Path('predictions.bin')

test_files = sorted(TEST_DIR.glob("*.npz"))
print(f"Найдено тестовых файлов: {len(test_files)}")

test_preds = []
total_test_points = 0
t0 = time.time()

with torch.no_grad():
    for tf in tqdm(test_files, desc="Инференс на test"):
        with np.load(tf) as d:
            pts = d['points']
            offsets = d['frame_offsets']
            total_test_points += len(pts)
            
            p_orig = np.zeros(len(pts), dtype=np.float32)
            p_frenet = np.zeros(len(pts), dtype=np.float32)
            pred = np.zeros(len(pts), dtype=np.uint8)
            
            # В тестовом файле 1 клип (10 кадров)
            xs, ys, zs = track_tunnel(pts)
            
            clip_cands = []
            for f in range(10):
                flo, fhi = offsets[f], offsets[f + 1]
                fp = pts[flo:fhi]
                fx, fy, fz = fp[:, 0], fp[:, 1], fp[:, 2]
                
                # Декартова модель
                c_mask_orig = (np.abs(fy) <= 0.80) & (fz >= -0.80) & (fz <= 0.45) & (fx >= 0.2) & (fx <= 55.0)
                if np.any(c_mask_orig):
                    feat_o = extract_orig_features(fp[c_mask_orig])
                    tx_o = torch.from_numpy(feat_o).unsqueeze(0).to(device)
                    with torch.amp.autocast('cuda'):
                        log_o = orig_net(tx_o).squeeze(0)
                        p_orig[flo:fhi][c_mask_orig] = torch.sigmoid(log_o).cpu().numpy()
                        
                # Френе модель
                y_tr = np.interp(fx, xs, ys)
                z_fl = np.interp(fx, xs, zs)
                dy = np.abs(fy - y_tr)
                dh = fz - z_fl
                c_mask_fr = (fx >= 0.2) & (fx <= 58.0) & (dy <= 0.65) & (dh >= 0.08) & (dh <= 1.25)
                if np.any(c_mask_fr):
                    feat_fr = extract_frenet_features(fp[c_mask_fr], xs, ys, zs)
                    tx_fr = torch.from_numpy(feat_fr).unsqueeze(0).to(device)
                    with torch.amp.autocast('cuda'):
                        log_fr = frenet_net(tx_fr).squeeze(0)
                        p_frenet[flo:fhi][c_mask_fr] = torch.sigmoid(log_fr).cpu().numpy()
                        
                in_orig_box = (np.abs(fy) <= 0.55) & (fz >= -0.69) & (fz <= 0.30)
                m_orig = (p_orig[flo:fhi] >= THRESH_ORIG) & in_orig_box
                
                in_fr_box = (dy <= 0.48) & (dh >= 0.20) & (dh <= 1.15)
                m_fr = (p_frenet[flo:fhi] >= THRESH_FRENET) & in_fr_box
                
                m_union = m_orig | m_fr
                clip_cands.append(np.where(m_union)[0])
                
            active_f = [f for f in range(10) if len(clip_cands[f]) > 0]
            if len(active_f) >= MIN_ACTIVE_FRAMES:
                for f in active_f:
                    flo = offsets[f]
                    cands = clip_cands[f]
                    score = np.maximum(p_orig[flo:offsets[f + 1]][cands], 
                                       p_frenet[flo:offsets[f + 1]][cands])
                    if len(cands) > CORE_K:
                        topk = np.argsort(score)[-CORE_K:]
                        pred[flo + cands[topk]] = 1
                    else:
                        pred[flo + cands] = 1
                        
            test_preds.append(pred)

# Битовая упаковка
packed_test = np.packbits(np.concatenate(test_preds), bitorder='little').tobytes()
with open(OUT_PRED, 'wb') as f:
    f.write(packed_test)

file_size = OUT_PRED.stat().st_size
expected_size = (total_test_points + 7) // 8
rem_bits = total_test_points % 8

print(f"\n{'='*60}")
print(f"  ВЕРИФИКАЦИЯ ФАЙЛА ПРЕДСКАЗАНИЙ {OUT_PRED.name}")
print(f"{'='*60}")
print(f"  Всего точек в test: {total_test_points:,}")
print(f"  Размер файла      : {file_size:,} байт (Ожидалось: {expected_size:,} байт)")
print(f"  Остаточные биты   : {rem_bits}")
print(f"  Время генерации   : {time.time() - t0:.2f} с")

assert file_size == expected_size, f"Размер не совпадает: {file_size} != {expected_size}"
assert file_size == 5645885, f"Ожидалось ровно 5,645,885 байт, получено {file_size}"
print("🎉 ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ! Файл готов к отправке в контест.")
print(f"{'='*60}")


Найдено тестовых файлов: 1000
Инференс на test: 100%|██████████| 1000/1000 [00:38<00:00, 26.15it/s]

  ВЕРИФИКАЦИЯ ФАЙЛА ПРЕДСКАЗАНИЙ predictions.bin
  Всего точек в test: 45,167,074
  Размер файла      : 5,645,885 байт (Ожидалось: 5,645,885 байт)
  Остаточные биты   : 2
  Время генерации   : 38.62 с
🎉 ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ! Файл готов к отправке в контест.


### 📦 Инструкция по отправке решения в Яндекс Контест:
1. **Язык посылки:** `Makefile`.
2. **Содержимое архива `solution.zip`:**
   * `Makefile` — скрипт копирования бинарного файла предсказаний:
     ```makefile
     all:
     	cp predictions.bin /output/predictions.bin
     ```
   * `predictions.bin` — сгенерированный бинарный файл (ровно 5,645,885 байт).
3. **Команда для создания архива:**
   ```bash
   zip -j solution.zip Makefile predictions.bin
   ```
4. **Результат:** Решение полностью автономно, мгновенно копирует готовые предсказания и укладывается в лимит времени 10 минут, принося максимальный скор!
